In [0]:
# =============================================================================
# LANDING ZONE SYNTHETIC DATA GENERATOR
# Databricks / PySpark
#
# No Window dependency
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
import datetime
import random


# =============================================================================
# CONFIGURATION
# =============================================================================

LANDING_PATH = "/Volumes/main/default/landing_zone"

RUN_DATE = datetime.date.today()
RUN_DATE_STR = RUN_DATE.strftime("%Y%m%d")
RUN_TS_STR = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

dbutils.widgets.dropdown(
    "is_incremental",
    "false",
    ["true", "false"]
)

IS_DAILY_INCREMENTAL = (
    dbutils.widgets.get("is_incremental") == "true"
)


# =============================================================================
# BASE (ONE-TIME HISTORICAL BOOTSTRAP) VOLUMES
# =============================================================================

NUM_CUSTOMERS = 99_685
NUM_ACCOUNTS = 149_596
NUM_MERCHANTS = 5_000
NUM_BILLERS = 286
NUM_INITIAL_TX_BACKFILL = 1_500_000
BACKFILL_HISTORY_DAYS = 379


# =============================================================================
# DAILY INCREMENTAL VOLUMES
# =============================================================================

NEW_CUSTOMERS_PER_DAY = 299
NEW_ACCOUNTS_PER_DAY = 381
DAILY_TX_VOLUME = 6_000


# =============================================================================
# TRANSACTION MIX
# =============================================================================

TX_TYPE_WEIGHTS = {
    "purchase": 0.65,
    "bill_payment": 0.20,
    "transfer": 0.15
}


# =============================================================================
# COMMISSION CONFIGURATION
# =============================================================================

COMMISSION_DAILY_CHANGE_FRACTION = 0.12


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def path(sub):
    return f"{LANDING_PATH}/{sub}"


def landing_exists(sub):
    """
    Check whether a landing-zone path exists.
    """

    try:
        dbutils.fs.ls(path(sub))
        return True

    except Exception as e:

        error_text = str(e).lower()

        if (
            "path does not exist" in error_text
            or "not found" in error_text
            or "filenotfound" in error_text
        ):
            return False

        raise


# =============================================================================
# 1. CUSTOMERS
# =============================================================================

def generate_customers():

    print("\n--- Processing CUSTOMERS ---")

    # =========================================================================
    # INITIAL LOAD
    # =========================================================================

    if not IS_DAILY_INCREMENTAL:

        df = (
            spark.range(1, NUM_CUSTOMERS + 1)

            .withColumn(
                "customer_id",
                F.concat(
                    F.lit("CUST_"),
                    F.lpad(
                        F.col("id"),
                        8,
                        "0"
                    )
                )
            )

            .withColumn(
                "first_name",
                F.element_at(
                    F.array(
                        F.lit("John"),
                        F.lit("Jane"),
                        F.lit("Alex"),
                        F.lit("Emily"),
                        F.lit("Michael")
                    ),
                    (F.rand() * 5 + 1).cast("int")
                )
            )

            .withColumn(
                "last_name",
                F.element_at(
                    F.array(
                        F.lit("Smith"),
                        F.lit("Doe"),
                        F.lit("Johnson"),
                        F.lit("Brown"),
                        F.lit("Taylor")
                    ),
                    (F.rand() * 5 + 1).cast("int")
                )
            )

            # Unique primary email
            .withColumn(
                "primary_email",
                F.lower(
                    F.concat(
                        F.col("first_name"),
                        F.lit("."),
                        F.col("last_name"),
                        F.lit("_"),
                        F.col("id"),
                        F.lit("@email.com")
                    )
                )
            )

            # Unique secondary email when present
            .withColumn(
                "secondary_email",
                F.when(
                    F.rand() > 0.7,
                    F.lower(
                        F.concat(
                            F.col("first_name"),
                            F.lit("_alt_"),
                            F.col("id"),
                            F.lit("@email.com")
                        )
                    )
                ).otherwise(
                    F.lit(None).cast("string")
                )
            )

            .withColumn(
                "phone_number",
                F.when(
                    F.rand() > 0.1,
                    F.concat(
                        F.lit("+1"),
                        (
                            F.rand() * 8_000_000_000
                            + 1_000_000_000
                        ).cast("long")
                    )
                ).otherwise(
                    F.lit(None).cast("string")
                )
            )

            .withColumn(
                "country_code",
                F.element_at(
                    F.array(
                        F.lit("US"),
                        F.lit("CA"),
                        F.lit("GB"),
                        F.lit("DE")
                    ),
                    (F.rand() * 4 + 1).cast("int")
                )
            )

            .withColumn(
                "credit_score",
                (F.rand() * 500 + 350).cast("int")
            )

            .withColumn(
                "kyc_status",
                F.when(
                    F.rand() > 0.05,
                    "VERIFIED"
                ).otherwise(
                    "PENDING"
                )
            )

            .withColumn(
                "signup_date",
                F.date_sub(
                    F.lit(RUN_DATE),
                    (F.rand() * BACKFILL_HISTORY_DAYS).cast("int")
                )
            )

            .withColumn(
                "created_at",
                F.to_timestamp(
                    F.col("signup_date")
                )
            )

            .withColumn(
                "updated_at",
                F.col("created_at")
            )

            .drop("id")
        )

        count = df.count()

        (
            df.write
            .format("parquet")
            .mode("overwrite")
            .save(path("customers"))
        )

        print(
            f"[CUSTOMERS] Initial historical backfill complete. "
            f"Wrote {count:,} records."
        )

        return

    # =========================================================================
    # DAILY NEW CUSTOMERS
    # =========================================================================

    df_new = (
        spark.range(1, NEW_CUSTOMERS_PER_DAY + 1)

        .withColumn(
            "customer_id",
            F.concat(
                F.lit("CUST_"),
                F.lit(RUN_TS_STR),
                F.lit("_"),
                F.lpad(
                    F.col("id"),
                    6,
                    "0"
                )
            )
        )

        .withColumn(
            "first_name",
            F.element_at(
                F.array(
                    F.lit("John"),
                    F.lit("Jane"),
                    F.lit("Alex"),
                    F.lit("Emily"),
                    F.lit("Michael")
                ),
                (F.rand() * 5 + 1).cast("int")
            )
        )

        .withColumn(
            "last_name",
            F.element_at(
                F.array(
                    F.lit("Smith"),
                    F.lit("Doe"),
                    F.lit("Johnson"),
                    F.lit("Brown"),
                    F.lit("Taylor")
                ),
                (F.rand() * 5 + 1).cast("int")
            )
        )

        .withColumn(
            "primary_email",
            F.lower(
                F.concat(
                    F.col("first_name"),
                    F.lit("."),
                    F.col("last_name"),
                    F.lit("_"),
                    F.lit(RUN_TS_STR),
                    F.lit("_"),
                    F.col("id"),
                    F.lit("@email.com")
                )
            )
        )

        .withColumn(
            "secondary_email",
            F.when(
                F.rand() > 0.7,
                F.lower(
                    F.concat(
                        F.col("first_name"),
                        F.lit("_alt_"),
                        F.lit(RUN_TS_STR),
                        F.lit("_"),
                        F.col("id"),
                        F.lit("@email.com")
                    )
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )

        .withColumn(
            "phone_number",
            F.when(
                F.rand() > 0.1,
                F.concat(
                    F.lit("+1"),
                    (
                        F.rand() * 8_000_000_000
                        + 1_000_000_000
                    ).cast("long")
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )

        .withColumn(
            "country_code",
            F.element_at(
                F.array(
                    F.lit("US"),
                    F.lit("CA"),
                    F.lit("GB"),
                    F.lit("DE")
                ),
                (F.rand() * 4 + 1).cast("int")
            )
        )

        .withColumn(
            "credit_score",
            (F.rand() * 500 + 350).cast("int")
        )

        .withColumn(
            "kyc_status",
            F.when(
                F.rand() > 0.15,
                "VERIFIED"
            ).otherwise(
                "PENDING"
            )
        )

        .withColumn(
            "signup_date",
            F.lit(RUN_DATE)
        )

        .withColumn(
            "created_at",
            F.current_timestamp()
        )

        .withColumn(
            "updated_at",
            F.current_timestamp()
        )

        .drop("id")
    )

    new_count = df_new.count()

    # =========================================================================
    # CUSTOMER UPDATES
    # =========================================================================

    df_updates = None
    update_count = 0

    if landing_exists("customers"):

        df_existing = (
            spark.read.parquet(
                path("customers")
            )
        )

        df_updates = (
            df_existing

            .sample(
                withReplacement=False,
                fraction=0.02
            )

            .withColumn(
                "credit_score",
                (F.rand() * 500 + 350).cast("int")
            )

            .withColumn(
                "secondary_email",
                F.when(
                    F.rand() > 0.7,
                    F.lower(
                        F.concat(
                            F.col("first_name"),
                            F.lit("_alt_"),
                            F.regexp_replace(
                                F.col("customer_id"),
                                "CUST_",
                                ""
                            ),
                            F.lit("@email.com")
                        )
                    )
                ).otherwise(
                    F.lit(None).cast("string")
                )
            )

            .withColumn(
                "kyc_status",
                F.when(
                    F.rand() > 0.02,
                    "VERIFIED"
                ).otherwise(
                    "PENDING"
                )
            )

            .withColumn(
                "updated_at",
                F.current_timestamp()
            )
        )

        update_count = df_updates.count()

    # =========================================================================
    # APPEND
    # =========================================================================

    if df_updates is None:
        df_out = df_new
    else:
        df_out = df_new.unionByName(
            df_updates
        )

    total_appended = df_out.count()

    (
        df_out.write
        .format("parquet")
        .mode("append")
        .save(path("customers"))
    )

    print(
        f"[CUSTOMERS] Incremental batch appended. "
        f"New customers: {new_count:,} | "
        f"Updated profile rows: {update_count:,} | "
        f"Total added: {total_appended:,}"
    )


# =============================================================================
# 2. ACCOUNTS
# =============================================================================

def generate_accounts():

    print("\n--- Processing ACCOUNTS ---")

    # =========================================================================
    # INITIAL LOAD
    # =========================================================================

    if not IS_DAILY_INCREMENTAL:

        df = (
            spark.range(1, NUM_ACCOUNTS + 1)

            .withColumn(
                "account_id",
                F.concat(
                    F.lit("ACC_"),
                    F.lpad(
                        F.col("id"),
                        8,
                        "0"
                    )
                )
            )

            # Valid historical customer IDs
            .withColumn(
                "customer_id",
                F.concat(
                    F.lit("CUST_"),
                    F.lpad(
                        (
                            F.rand()
                            * NUM_CUSTOMERS
                            + 1
                        ).cast("int"),
                        8,
                        "0"
                    )
                )
            )

            .withColumn(
                "account_type",
                F.when(
                    F.rand() > 0.3,
                    "Checking"
                ).otherwise(
                    "Savings"
                )
            )

            .withColumn(
                "account_status",
                F.when(
                    F.rand() > 0.25,
                    "Active"
                ).otherwise(
                    "Dormant"
                )
            )

            .withColumn(
                "currency",
                F.lit("USD")
            )

            .withColumn(
                "created_at",
                F.to_timestamp(
                    F.date_sub(
                        F.lit(RUN_DATE),
                        (
                            F.rand()
                            * BACKFILL_HISTORY_DAYS
                        ).cast("int")
                    )
                )
            )

            .withColumn(
                "updated_at",
                F.col("created_at")
            )

            .drop("id")
        )

        count = df.count()

        (
            df.write
            .format("parquet")
            .mode("overwrite")
            .save(path("accounts"))
        )

        print(
            f"[ACCOUNTS] Initial historical backfill complete. "
            f"Wrote {count:,} records."
        )

        return

    # =========================================================================
    # DAILY NEW ACCOUNTS
    #
    # We deliberately avoid Window.
    #
    # For the first 299 accounts, assign today's newly-created customers.
    # For the remaining 82 accounts, assign valid historical customers.
    # =========================================================================

    df_new = (
        spark.range(1, NEW_ACCOUNTS_PER_DAY + 1)

        .withColumn(
            "account_id",
            F.concat(
                F.lit("ACC_"),
                F.lit(RUN_TS_STR),
                F.lit("_"),
                F.lpad(
                    F.col("id"),
                    6,
                    "0"
                )
            )
        )

        .withColumn(
            "customer_id",
            F.when(
                F.col("id") <= NEW_CUSTOMERS_PER_DAY,

                # Today's new customer
                F.concat(
                    F.lit("CUST_"),
                    F.lit(RUN_TS_STR),
                    F.lit("_"),
                    F.lpad(
                        F.col("id"),
                        6,
                        "0"
                    )
                )
            ).otherwise(

                # Valid historical customer
                F.concat(
                    F.lit("CUST_"),
                    F.lpad(
                        (
                            (
                                F.rand()
                                * NUM_CUSTOMERS
                            ).cast("int")
                            + 1
                        ),
                        8,
                        "0"
                    )
                )
            )
        )

        .withColumn(
            "account_type",
            F.when(
                F.rand() > 0.3,
                "Checking"
            ).otherwise(
                "Savings"
            )
        )

        .withColumn(
            "account_status",
            F.lit("Active")
        )

        .withColumn(
            "currency",
            F.lit("USD")
        )

        .withColumn(
            "created_at",
            F.current_timestamp()
        )

        .withColumn(
            "updated_at",
            F.current_timestamp()
        )

        .drop("id")
    )

    new_count = df_new.count()

    # =========================================================================
    # DAILY ACCOUNT STATUS UPDATES
    # =========================================================================

    df_status_updates = None
    update_count = 0

    if landing_exists("accounts"):

        df_existing = (
            spark.read.parquet(
                path("accounts")
            )
        )

        df_status_updates = (
            df_existing

            .sample(
                withReplacement=False,
                fraction=0.01
            )

            .withColumn(
                "account_status",
                F.when(
                    F.col("account_status") == "Active",
                    "Dormant"
                ).otherwise(
                    "Active"
                )
            )

            .withColumn(
                "updated_at",
                F.current_timestamp()
            )
        )

        update_count = df_status_updates.count()

    # =========================================================================
    # APPEND
    # =========================================================================

    if df_status_updates is None:
        df_out = df_new
    else:
        df_out = df_new.unionByName(
            df_status_updates
        )

    total_appended = df_out.count()

    (
        df_out.write
        .format("parquet")
        .mode("append")
        .save(path("accounts"))
    )

    print(
        f"[ACCOUNTS] Incremental batch appended. "
        f"New accounts: {new_count:,} | "
        f"Account status changes: {update_count:,} | "
        f"Total added: {total_appended:,}"
    )


# =============================================================================
# 3. MERCHANTS
# =============================================================================

def generate_merchants():

    print("\n--- Processing MERCHANTS ---")

    if IS_DAILY_INCREMENTAL:

        print(
            "[MERCHANTS] Incremental run - "
            "static reference table skipped."
        )

        return

    df = (
        spark.range(1, NUM_MERCHANTS + 1)

        .withColumn(
            "merchant_id",
            F.concat(
                F.lit("MERCH_"),
                F.lpad(
                    F.col("id"),
                    6,
                    "0"
                )
            )
        )

        .withColumn(
            "merchant_name",
            F.concat(
                F.lit("Store_"),
                F.col("id")
            )
        )

        .withColumn(
            "mcc_code",
            F.element_at(
                F.array(
                    F.lit("5411"),
                    F.lit("5812"),
                    F.lit("5732"),
                    F.lit("5999")
                ),
                (F.rand() * 4 + 1).cast("int")
            )
        )

        .withColumn(
            "country_code",
            F.element_at(
                F.array(
                    F.lit("US"),
                    F.lit("CA"),
                    F.lit("GB"),
                    F.lit("DE")
                ),
                (F.rand() * 4 + 1).cast("int")
            )
        )

        .withColumn(
            "onboarded_at",
            F.date_sub(
                F.lit(RUN_DATE),
                (
                    F.rand()
                    * BACKFILL_HISTORY_DAYS
                ).cast("int")
            )
        )

        .drop("id")
    )

    count = df.count()

    (
        df.write
        .format("csv")
        .option("header", "true")
        .mode("overwrite")
        .save(path("merchants"))
    )

    print(
        f"[MERCHANTS] Wrote {count:,} merchant dimension records."
    )


# =============================================================================
# 4. BILLERS
# =============================================================================

def generate_billers():

    print("\n--- Processing BILLERS ---")

    if IS_DAILY_INCREMENTAL:

        print(
            "[BILLERS] Incremental run - "
            "static reference table skipped."
        )

        return

    df = (
        spark.range(1, NUM_BILLERS + 1)

        .withColumn(
            "biller_id",
            F.concat(
                F.lit("BILL_"),
                F.lpad(
                    F.col("id"),
                    5,
                    "0"
                )
            )
        )

        .withColumn(
            "biller_name",
            F.concat(
                F.lit("Biller_"),
                F.col("id")
            )
        )

        .withColumn(
            "biller_category",
            F.element_at(
                F.array(
                    F.lit("Electricity"),
                    F.lit("Water"),
                    F.lit("Internet"),
                    F.lit("Mobile_Topup"),
                    F.lit("Insurance"),
                    F.lit("Education")
                ),
                (F.rand() * 6 + 1).cast("int")
            )
        )

        .withColumn(
            "country_code",
            F.element_at(
                F.array(
                    F.lit("US"),
                    F.lit("CA"),
                    F.lit("GB"),
                    F.lit("DE")
                ),
                (F.rand() * 4 + 1).cast("int")
            )
        )

        .drop("id")
    )

    count = df.count()

    (
        df.write
        .format("csv")
        .option("header", "true")
        .mode("overwrite")
        .save(path("billers"))
    )

    print(
        f"[BILLERS] Wrote {count:,} biller dimension records."
    )


# =============================================================================
# 5. COMMISSION RULES
# =============================================================================

COMMISSION_SCOPES = [
    ("purchase", "5411"),
    ("purchase", "5812"),
    ("purchase", "5732"),
    ("purchase", "5999"),
    ("bill_payment", "Electricity"),
    ("bill_payment", "Water"),
    ("bill_payment", "Internet"),
    ("bill_payment", "Mobile_Topup"),
    ("bill_payment", "Insurance"),
    ("bill_payment", "Education"),
    ("transfer", "internal"),
]


def generate_commission_rules():

    print("\n--- Processing COMMISSION RULES ---")

    # =========================================================================
    # INITIAL LOAD
    # =========================================================================

    if not IS_DAILY_INCREMENTAL:

        rows = []

        for tx_type, scope_value in COMMISSION_SCOPES:

            commission_type = (
                "percentage"
                if tx_type != "transfer"
                else "flat"
            )

            commission_value = (
                round(
                    random.uniform(
                        0.010,
                        0.035
                    ),
                    4
                )
                if tx_type != "transfer"
                else 0.50
            )

            rows.append(
                (
                    f"RULE_{tx_type}_{scope_value}",
                    tx_type,
                    scope_value,
                    commission_type,
                    float(commission_value)
                )
            )

        schema = StructType(
            [
                StructField(
                    "commission_rule_id",
                    StringType(),
                    True
                ),
                StructField(
                    "scope_type",
                    StringType(),
                    True
                ),
                StructField(
                    "scope_value",
                    StringType(),
                    True
                ),
                StructField(
                    "commission_type",
                    StringType(),
                    True
                ),
                StructField(
                    "commission_value",
                    DoubleType(),
                    True
                )
            ]
        )

        df = (
            spark.createDataFrame(
                rows,
                schema
            )
            .withColumn(
                "updated_at",
                F.current_timestamp()
            )
        )

        count = df.count()

        (
            df.write
            .format("parquet")
            .mode("overwrite")
            .save(path("commission_rules"))
        )

        print(
            f"[COMMISSION RULES] Initial setup complete. "
            f"Wrote {count:,} rules."
        )

        return

    # =========================================================================
    # DAILY REPRICING
    # =========================================================================

    df_existing = (
        spark.read.parquet(
            path("commission_rules")
        )
    )

    df_out = (
        df_existing

        .withColumn(
            "_reprice",
            F.rand()
            < COMMISSION_DAILY_CHANGE_FRACTION
        )

        .withColumn(
            "commission_value",
            F.when(
                F.col("_reprice"),

                F.round(
                    F.col("commission_value")
                    * (
                        F.lit(1)
                        + (
                            F.rand() * 0.4
                            - 0.2
                        )
                    ),
                    4
                )

            ).otherwise(
                F.col("commission_value")
            )
        )

        .withColumn(
            "updated_at",
            F.when(
                F.col("_reprice"),
                F.current_timestamp()
            ).otherwise(
                F.col("updated_at")
            )
        )
    )

    repriced_count = (
        df_out
        .filter(F.col("_reprice"))
        .count()
    )

    total_count = df_out.count()

    (
        df_out
        .drop("_reprice")
        .write
        .format("parquet")
        .mode("overwrite")
        .save(path("commission_rules"))
    )

    print(
        f"[COMMISSION RULES] Overwrote state table. "
        f"Total active rules: {total_count:,} | "
        f"Rules re-priced today: {repriced_count:,}"
    )


# =============================================================================
# 6. TRANSACTIONS
# =============================================================================

def _new_transactions(
    n_rows,
    id_prefix_seed,
    spread_over_history=False
):

    # =========================================================================
    # BASE TRANSACTION DATA
    # =========================================================================

    df = (
        spark.range(1, n_rows + 1)

        .withColumn(
            "transaction_id",
            F.concat(
                F.lit("TX_"),
                F.lit(id_prefix_seed),
                F.lit("_"),
                F.lpad(
                    F.col("id"),
                    8,
                    "0"
                )
            )
        )

        .withColumn(
            "_r",
            F.rand()
        )
    )

    # =========================================================================
    # TRANSACTION TYPE
    # =========================================================================

    df = (
        df

        .withColumn(
            "transaction_type",

            F.when(
                F.col("_r")
                < TX_TYPE_WEIGHTS["purchase"],
                "purchase"
            )

            .when(
                F.col("_r")
                < (
                    TX_TYPE_WEIGHTS["purchase"]
                    + TX_TYPE_WEIGHTS["bill_payment"]
                ),
                "bill_payment"
            )

            .otherwise(
                "transfer"
            )
        )

        .drop("_r")
    )

    # =========================================================================
    # VALID ACCOUNT IDs
    #
    # Historical accounts:
    # ACC_00000001 through ACC_149596
    #
    # Today's incremental accounts:
    # ACC_<RUN_TS_STR>_000001 through ACC_<RUN_TS_STR>_000381
    #
    # We generate IDs from these known-valid ranges.
    # =========================================================================

    df = (
        df

        .withColumn(
            "_account_choice",
            F.rand()
        )

        .withColumn(
            "account_id",

            F.when(
                F.col("_account_choice") < 0.98,

                # Historical account
                F.concat(
                    F.lit("ACC_"),
                    F.lpad(
                        (
                            F.rand()
                            * NUM_ACCOUNTS
                            + 1
                        ).cast("int"),
                        8,
                        "0"
                    )
                )

            ).otherwise(

                # Today's new account
                F.concat(
                    F.lit("ACC_"),
                    F.lit(RUN_TS_STR),
                    F.lit("_"),
                    F.lpad(
                        (
                            F.rand()
                            * NEW_ACCOUNTS_PER_DAY
                            + 1
                        ).cast("int"),
                        6,
                        "0"
                    )
                )
            )
        )

        .drop("_account_choice")
    )

    # =========================================================================
    # DESTINATION ACCOUNT
    # =========================================================================

    df = (
        df

        .withColumn(
            "_destination_choice",
            F.rand()
        )

        .withColumn(
            "_destination_account",

            F.when(
                F.col("_destination_choice") < 0.98,

                F.concat(
                    F.lit("ACC_"),
                    F.lpad(
                        (
                            F.rand()
                            * NUM_ACCOUNTS
                            + 1
                        ).cast("int"),
                        8,
                        "0"
                    )
                )

            ).otherwise(

                F.concat(
                    F.lit("ACC_"),
                    F.lit(RUN_TS_STR),
                    F.lit("_"),
                    F.lpad(
                        (
                            F.rand()
                            * NEW_ACCOUNTS_PER_DAY
                            + 1
                        ).cast("int"),
                        6,
                        "0"
                    )
                )
            )
        )

        .withColumn(
            "destination_account_id",

            F.when(
                F.col("transaction_type") == "transfer",
                F.col("_destination_account")
            ).otherwise(
                F.lit(None).cast("string")
            )
        )

        .drop(
            "_destination_choice",
            "_destination_account"
        )
    )

    # =========================================================================
    # MERCHANT
    #
    # Only purchase transactions receive merchant IDs.
    # =========================================================================

    df = (
        df

        .withColumn(
            "merchant_id",

            F.when(
                F.col("transaction_type") == "purchase",

                F.concat(
                    F.lit("MERCH_"),
                    F.lpad(
                        (
                            F.rand()
                            * NUM_MERCHANTS
                            + 1
                        ).cast("int"),
                        6,
                        "0"
                    )
                )

            ).otherwise(
                F.lit(None).cast("string")
            )
        )
    )

    # =========================================================================
    # BILLER
    #
    # Only bill-payment transactions receive biller IDs.
    # =========================================================================

    df = (
        df

        .withColumn(
            "biller_id",

            F.when(
                F.col("transaction_type")
                == "bill_payment",

                F.concat(
                    F.lit("BILL_"),
                    F.lpad(
                        (
                            F.rand()
                            * NUM_BILLERS
                            + 1
                        ).cast("int"),
                        5,
                        "0"
                    )
                )

            ).otherwise(
                F.lit(None).cast("string")
            )
        )

        .withColumn(
            "biller_reference_number",

            F.when(
                F.col("transaction_type")
                == "bill_payment",

                F.concat(
                    F.lit("REF"),
                    (
                        F.rand()
                        * 900_000_000
                        + 100_000_000
                    ).cast("long")
                )

            ).otherwise(
                F.lit(None).cast("string")
            )
        )
    )

    # =========================================================================
    # AMOUNT
    # =========================================================================

    df = (
        df

        .withColumn(
            "amount",

            F.when(
                F.col("transaction_type")
                == "purchase",

                F.round(
                    F.rand() * 1200 + 1.50,
                    2
                )
            )

            .when(
                F.col("transaction_type")
                == "bill_payment",

                F.round(
                    F.rand() * 300 + 10.00,
                    2
                )
            )

            .otherwise(

                F.round(
                    F.rand() * 5000 + 5.00,
                    2
                )
            )
        )
    )

    # =========================================================================
    # CHANNEL
    # =========================================================================

    df = (
        df

        .withColumn(
            "channel",

            F.when(
                F.col("transaction_type")
                == "purchase",

                F.element_at(
                    F.array(
                        F.lit("Web"),
                        F.lit("Mobile"),
                        F.lit("POS"),
                        F.lit("ATM")
                    ),
                    (F.rand() * 4 + 1).cast("int")
                )
            )

            .otherwise(

                F.element_at(
                    F.array(
                        F.lit("Web"),
                        F.lit("Mobile")
                    ),
                    (F.rand() * 2 + 1).cast("int")
                )
            )
        )
    )

    # =========================================================================
    # CARD NUMBER
    # =========================================================================

    df = (
        df

        .withColumn(
            "card_number",

            F.when(
                (
                    F.col("transaction_type")
                    == "purchase"
                )
                &
                (
                    F.col("channel")
                    .isin("Web", "POS")
                ),

                F.concat(
                    F.lit("4532xxxxxx"),
                    F.lpad(
                        (
                            F.rand()
                            * 9999
                        ).cast("int"),
                        4,
                        "0"
                    )
                )

            ).otherwise(
                F.lit(None).cast("string")
            )
        )
    )

    # =========================================================================
    # IP ADDRESS
    # =========================================================================

    df = (
        df

        .withColumn(
            "ip_address",

            F.when(
                F.col("channel")
                .isin("Web", "Mobile"),

                F.concat(
                    (
                        F.rand() * 200 + 1
                    ).cast("int"),

                    F.lit("."),

                    (
                        F.rand() * 200 + 1
                    ).cast("int"),

                    F.lit("."),

                    (
                        F.rand() * 200 + 1
                    ).cast("int"),

                    F.lit("."),

                    (
                        F.rand() * 200 + 1
                    ).cast("int")
                )

            ).otherwise(
                F.lit(None).cast("string")
            )
        )
    )

    # =========================================================================
    # CURRENCY
    # =========================================================================

    df = df.withColumn(
        "currency",
        F.lit("USD")
    )

    # =========================================================================
    # DATE DISTRIBUTION
    # =========================================================================

    if spread_over_history:

        df = df.withColumn(
            "_days_ago",
            (
                F.rand()
                * BACKFILL_HISTORY_DAYS
            ).cast("int")
        )

    else:

        df = df.withColumn(
            "_days_ago",
            F.lit(0)
        )

    df = (
        df

        .withColumn(
            "_seconds_in_day",
            (
                F.rand()
                * 86400
            ).cast("int")
        )

        .withColumn(
            "_status_r",
            F.rand()
        )
    )

    base_unix = F.unix_timestamp(
        F.lit(str(RUN_DATE)),
        "yyyy-MM-dd"
    )

    # =========================================================================
    # CREATED AT
    # =========================================================================

    df = (
        df

        .withColumn(
            "created_at",

            F.from_unixtime(
                base_unix
                - (
                    F.col("_days_ago")
                    * 86400
                )
                + F.col("_seconds_in_day")
            ).cast("timestamp")
        )
    )

    # =========================================================================
    # STATUS
    # =========================================================================

    df = (
        df

        .withColumn(
            "status",

            F.when(
                F.col("_days_ago") == 0,

                F.when(
                    F.col("_status_r") > 0.20,
                    "SETTLED"
                ).otherwise(
                    "PENDING"
                )
            )

            .otherwise(

                F.when(
                    F.col("_status_r") < 0.90,
                    "SETTLED"
                )

                .when(
                    F.col("_status_r") < 0.96,
                    "DECLINED"
                )

                .otherwise(
                    "CANCELLED"
                )
            )
        )

        .withColumn(
            "updated_at",
            F.col("created_at")
        )

        .drop(
            "_days_ago",
            "_seconds_in_day",
            "_status_r"
        )
    )

    return df


# =============================================================================
# GENERATE TRANSACTIONS
# =============================================================================

def generate_transactions():

    print("\n--- Processing TRANSACTIONS ---")

    # =========================================================================
    # INITIAL HISTORICAL LOAD
    # =========================================================================

    if not IS_DAILY_INCREMENTAL:

        df_tx = _new_transactions(
            n_rows=NUM_INITIAL_TX_BACKFILL,
            id_prefix_seed=RUN_TS_STR,
            spread_over_history=True
        )

        count = df_tx.count()

        (
            df_tx.write
            .format("json")
            .mode("overwrite")
            .save(path("transactions"))
        )

        print(
            f"[TRANSACTIONS] Initial historical backfill complete. "
            f"Wrote {count:,} raw transactions over "
            f"{BACKFILL_HISTORY_DAYS} days."
        )

        return

    # =========================================================================
    # DAILY TRANSACTIONS
    # =========================================================================

    df_new = _new_transactions(
        n_rows=DAILY_TX_VOLUME,
        id_prefix_seed=RUN_TS_STR,
        spread_over_history=False
    )

    new_count = df_new.count()

    # =========================================================================
    # RESOLVE PENDING TRANSACTIONS
    # =========================================================================

    df_pending_updates = None
    pending_count = 0

    if landing_exists("transactions"):

        df_all = (
            spark.read.json(
                path("transactions")
            )
        )

        df_pending = (
            df_all
            .filter(
                F.col("status") == "PENDING"
            )
        )

        pending_exists = (
            df_pending
            .limit(1)
            .count()
            > 0
        )

        if pending_exists:

            df_pending_updates = (
                df_pending

                .withColumn(
                    "status",

                    F.element_at(
                        F.array(
                            F.lit("SETTLED"),
                            F.lit("DECLINED"),
                            F.lit("CANCELLED")
                        ),
                        (
                            F.rand() * 3 + 1
                        ).cast("int")
                    )
                )

                .withColumn(
                    "updated_at",
                    F.current_timestamp()
                )
            )

            pending_count = (
                df_pending_updates.count()
            )

    # =========================================================================
    # APPEND
    # =========================================================================

    if df_pending_updates is None:

        df_out = df_new

    else:

        df_out = (
            df_new
            .unionByName(
                df_pending_updates
            )
        )

    total_appended = df_out.count()

    (
        df_out.write
        .format("json")
        .mode("append")
        .save(path("transactions"))
    )

    print(
        f"[TRANSACTIONS] Incremental batch appended. "
        f"New raw events today: {new_count:,} | "
        f"PENDING transactions resolved: {pending_count:,} | "
        f"Total added: {total_appended:,}"
    )


# =============================================================================
# RUN ENGINE
# =============================================================================

if __name__ == "__main__":

    print(
        "================================================================="
    )

    print(
        f" STARTING LANDING ZONE GENERATOR | "
        f"IS_INCREMENTAL={IS_DAILY_INCREMENTAL}"
    )

    print(
        f" DATE: {RUN_DATE} | "
        f"PATH: {LANDING_PATH}"
    )

    print(
        "================================================================="
    )

    generate_customers()
    generate_accounts()
    generate_merchants()
    generate_billers()
    generate_commission_rules()
    generate_transactions()

    print(
        "\n================================================================="
    )

    print(
        f" SUCCESS: Landing zone refreshed at "
        f"{LANDING_PATH}"
    )

    print(
        "================================================================="
    )


In [0]:
# %sql
# DROP TABLE IF EXISTS main.bronze.raw_transactions;
# DROP TABLE IF EXISTS main.bronze.raw_accounts;
# DROP TABLE IF EXISTS main.bronze.raw_customers;
# DROP TABLE IF EXISTS main.bronze.raw_merchants;
# DROP TABLE IF EXISTS main.bronze.raw_billers;
# DROP TABLE IF EXISTS main.bronze.raw_commission_rules;

In [0]:
# %sql
# -- =============================================================================
# -- FULL DATA ENVIRONMENT TEARDOWN (Databricks %sql Cell)
# -- =============================================================================

# -- 1. BRONZE LAYER TABLES
# DROP TABLE IF EXISTS main.bronze.raw_transactions;
# DROP TABLE IF EXISTS main.bronze.raw_accounts;
# DROP TABLE IF EXISTS main.bronze.raw_customers;
# DROP TABLE IF EXISTS main.bronze.raw_merchants;
# DROP TABLE IF EXISTS main.bronze.raw_billers;
# DROP TABLE IF EXISTS main.bronze.raw_commission_rules;

# -- 2. STAGING VIEWS
# DROP VIEW IF EXISTS stg_accounts;
# DROP VIEW IF EXISTS stg_billers;
# DROP VIEW IF EXISTS stg_customers;
# DROP VIEW IF EXISTS stg_merchants;
# DROP VIEW IF EXISTS stg_transactions;

# -- 3. MEDALLION TABLES & VIEWS
# DROP TABLE IF EXISTS silver;
# DROP TABLE IF EXISTS gold;
# DROP TABLE IF EXISTS bronze;

# -- 4. METADATA TABLES (If created manually)
# DROP TABLE IF EXISTS table_constraints;
# DROP TABLE IF EXISTS table_privileges;
# DROP TABLE IF EXISTS table_tags;
# DROP TABLE IF EXISTS tables;
# DROP TABLE IF EXISTS views;
# DROP TABLE IF EXISTS volume_privileges;
# DROP TABLE IF EXISTS volume_tags;
# DROP TABLE IF EXISTS volumes;
# DROP TABLE IF EXISTS schemata;

# -- 5. DROP SCHEMAS / DATABASES (CASCADE deletes all contained objects)
# DROP SCHEMA IF EXISTS bronze CASCADE;
# DROP SCHEMA IF EXISTS silver CASCADE;
# DROP SCHEMA IF EXISTS gold CASCADE;
# DROP SCHEMA IF EXISTS assignmentscala CASCADE;
# DROP SCHEMA IF EXISTS rowan_depi CASCADE;

# -- -- 6. DROP CATALOGS
# -- DROP CATALOG IF EXISTS depicatalog CASCADE;